In [1]:
import kagglehub

In [2]:
path = kagglehub.dataset_download("wyattowalsh/basketball")

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

# Paths
base_path = Path(path) / "csv"
clean_dir = Path.cwd() / "cleaned_csv"
clean_dir.mkdir(exist_ok=True)


def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
                  .str.lower()
                  .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
                  .str.replace(r"_+", "_", regex=True)
                  .str.strip("_")
    )
    return df

def to_int_safe(series):
    return pd.to_numeric(series, errors="coerce").astype("Int64")

def to_float_safe(series):
    return pd.to_numeric(series, errors="coerce")

def save_csv(df, filename):
    out_path = clean_dir / filename
    df.to_csv(out_path, index=False)
    print(f"Saved {filename} | shape={df.shape}")

# Load raw CSVs
players_raw = pd.read_csv(base_path / "player.csv")
teams_raw = pd.read_csv(base_path / "team.csv")
games_raw = pd.read_csv(base_path / "game.csv")

# -------------------------------------------------
# Clean players
# -------------------------------------------------
players = clean_columns(players_raw).drop_duplicates()

players = players[["id", "full_name", "first_name", "last_name", "is_active"]].copy()
players = players.dropna(subset=["id", "full_name"])

players["id"] = players["id"].astype(str).str.strip()
players["full_name"] = players["full_name"].astype(str).str.strip()
players["first_name"] = players["first_name"].astype(str).str.strip()
players["last_name"] = players["last_name"].astype(str).str.strip()
players["is_active"] = pd.to_numeric(players["is_active"], errors="coerce").fillna(0).astype(int)

players = players.rename(columns={
    "id": "player_id",
    "full_name": "player_name"
})

save_csv(players, "players_clean.csv")

# Clean teams

teams = clean_columns(teams_raw).drop_duplicates()

teams = teams[["id", "full_name", "abbreviation", "nickname", "city", "state", "year_founded"]].copy()
teams = teams.dropna(subset=["id", "full_name"])

teams["id"] = teams["id"].astype(str).str.strip()
teams["full_name"] = teams["full_name"].astype(str).str.strip()
teams["abbreviation"] = teams["abbreviation"].astype(str).str.strip()
teams["nickname"] = teams["nickname"].astype(str).str.strip()
teams["city"] = teams["city"].astype(str).str.strip()
teams["state"] = teams["state"].astype(str).str.strip()
teams["year_founded"] = to_int_safe(teams["year_founded"])

teams = teams.rename(columns={
    "id": "team_id",
    "full_name": "team_name"
})

save_csv(teams, "teams_clean.csv")

# -------------------------------------------------
# Clean games
# -------------------------------------------------
games = clean_columns(games_raw).drop_duplicates(subset=["game_id"])

games["game_date"] = pd.to_datetime(games["game_date"], errors="coerce").dt.date

games_clean = games[[
    "game_id", "season_id", "game_date", "season_type",
    "team_id_home", "team_id_away",
    "team_name_home", "team_name_away",
    "team_abbreviation_home", "team_abbreviation_away",
    "pts_home", "pts_away",
    "wl_home", "wl_away"
]].copy()

games_clean = games_clean.dropna(subset=["game_id", "season_id", "game_date"])

games_clean["game_id"] = games_clean["game_id"].astype(str).str.strip()
games_clean["season_id"] = games_clean["season_id"].astype(str).str.strip()
games_clean["season_type"] = games_clean["season_type"].astype(str).str.strip()

games_clean["team_id_home"] = pd.to_numeric(games_clean["team_id_home"], errors="coerce").astype("Int64")
games_clean["team_id_away"] = pd.to_numeric(games_clean["team_id_away"], errors="coerce").astype("Int64")
games_clean["pts_home"] = to_int_safe(games_clean["pts_home"])
games_clean["pts_away"] = to_int_safe(games_clean["pts_away"])
games_clean["wl_home"] = games_clean["wl_home"].astype(str).str.strip()
games_clean["wl_away"] = games_clean["wl_away"].astype(str).str.strip()

save_csv(games_clean, "games_clean.csv")

# -------------------------------------------------
# Create team_game_stats (normalized from home/away columns)
# -------------------------------------------------
common_base = games[["game_id", "season_id", "game_date", "season_type"]].copy()
common_base["game_id"] = common_base["game_id"].astype(str).str.strip()
common_base["season_id"] = common_base["season_id"].astype(str).str.strip()
common_base["game_date"] = pd.to_datetime(common_base["game_date"], errors="coerce").dt.date
common_base["season_type"] = common_base["season_type"].astype(str).str.strip()

home = common_base.copy()
home["team_id"] = games["team_id_home"]
home["team_abbreviation"] = games["team_abbreviation_home"]
home["team_name"] = games["team_name_home"]
home["wl"] = games["wl_home"]
home["fgm"] = games["fgm_home"]
home["fga"] = games["fga_home"]
home["fg_pct"] = games["fg_pct_home"]
home["fg3m"] = games["fg3m_home"]
home["fg3a"] = games["fg3a_home"]
home["fg3_pct"] = games["fg3_pct_home"]
home["ftm"] = games["ftm_home"]
home["fta"] = games["fta_home"]
home["ft_pct"] = games["ft_pct_home"]
home["oreb"] = games["oreb_home"]
home["dreb"] = games["dreb_home"]
home["reb"] = games["reb_home"]
home["ast"] = games["ast_home"]
home["stl"] = games["stl_home"]
home["blk"] = games["blk_home"]
home["tov"] = games["tov_home"]
home["pf"] = games["pf_home"]
home["pts"] = games["pts_home"]
home["plus_minus"] = games["plus_minus_home"]
home["video_available"] = games["video_available_home"]
home["is_home"] = 1

away = common_base.copy()
away["team_id"] = games["team_id_away"]
away["team_abbreviation"] = games["team_abbreviation_away"]
away["team_name"] = games["team_name_away"]
away["wl"] = games["wl_away"]
away["fgm"] = games["fgm_away"]
away["fga"] = games["fga_away"]
away["fg_pct"] = games["fg_pct_away"]
away["fg3m"] = games["fg3m_away"]
away["fg3a"] = games["fg3a_away"]
away["fg3_pct"] = games["fg3_pct_away"]
away["ftm"] = games["ftm_away"]
away["fta"] = games["fta_away"]
away["ft_pct"] = games["ft_pct_away"]
away["oreb"] = games["oreb_away"]
away["dreb"] = games["dreb_away"]
away["reb"] = games["reb_away"]
away["ast"] = games["ast_away"]
away["stl"] = games["stl_away"]
away["blk"] = games["blk_away"]
away["tov"] = games["tov_away"]
away["pf"] = games["pf_away"]
away["pts"] = games["pts_away"]
away["plus_minus"] = games["plus_minus_away"]
away["video_available"] = games["video_available_away"]
away["is_home"] = 0

team_game_stats = pd.concat([home, away], ignore_index=True)

team_game_stats = team_game_stats.dropna(subset=["game_id", "team_id", "pts"])
team_game_stats["team_id"] = pd.to_numeric(team_game_stats["team_id"], errors="coerce").astype("Int64")
team_game_stats["team_abbreviation"] = team_game_stats["team_abbreviation"].astype(str).str.strip()
team_game_stats["team_name"] = team_game_stats["team_name"].astype(str).str.strip()
team_game_stats["wl"] = team_game_stats["wl"].astype(str).str.strip()

numeric_cols = [
    "fgm", "fga", "fg_pct", "fg3m", "fg3a", "fg3_pct",
    "ftm", "fta", "ft_pct", "oreb", "dreb", "reb",
    "ast", "stl", "blk", "tov", "pf", "pts",
    "plus_minus", "video_available", "is_home"
]
for col in numeric_cols:
    team_game_stats[col] = pd.to_numeric(team_game_stats[col], errors="coerce")

team_game_stats = team_game_stats.drop_duplicates(subset=["game_id", "team_id", "is_home"])

save_csv(team_game_stats, "team_game_stats_clean.csv")

print("\nAll cleaned CSVs created successfully.")
print(f"Folder: {clean_dir}")

Saved players_clean.csv | shape=(4831, 5)
Saved teams_clean.csv | shape=(30, 7)
Saved games_clean.csv | shape=(65642, 14)
Saved team_game_stats_clean.csv | shape=(131284, 29)

All cleaned CSVs created successfully.
Folder: /Users/vivianwu/Desktop/applied database/nba_project/cleaned_csv


In [8]:
import pandas as pd

df = pd.read_csv("cleaned_csv/games_clean.csv")

# remove any accidental bad columns
df = df.loc[:, ~df.columns.astype(str).str.contains(r"^Unnamed|^None$", case=False, regex=True)]
df.columns = [str(c).strip() for c in df.columns]

# save again in a clean format
df.to_csv("cleaned_csv/games_clean.csv", index=False, encoding="utf-8")
print(df.columns.tolist())

['game_id', 'season_id', 'game_date', 'season_type', 'team_id_home', 'team_id_away', 'team_name_home', 'team_name_away', 'team_abbreviation_home', 'team_abbreviation_away', 'pts_home', 'pts_away', 'wl_home', 'wl_away']
